In [10]:
!pip uninstall -y vllm transformers accelerate bitsandbytes openai -q



In [ ]:
!pip install -q vllm==0.6.1.post2 transformers==4.44.2 accelerate==0.34.2 bitsandbytes==0.44.1 openai==1.47.0

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.7/170.7 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.4/122.4 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.6/375.6 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 117.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.9/75.9 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 80.2 MB/s eta 0:

# 8.4절 실습: LLM 서빙 프레임워크

## 예제 8.1. 실습에 사용할 데이터셋 준비

In [6]:
import torch
from datasets import load_dataset

def make_prompt(ddl, question, query=''):
    prompt = f"당신은 SQL을 생성하는 SQL 봇입니다. DDL의 테이블을 활용한 Question을 해결할 수 있는 SQL 쿼리를 생성하세요.\n### DDL:{ddl}\n### Question:{question}\n### SQL:{query}"
    return prompt

dataset = load_dataset("shangrilar/ko_text2sql", "origin")['test'].to_pandas()
dataset['prompt'] = [make_prompt(r['context'], r['question']) for _, r in dataset.iterrows()]

print(f"✅ 데이터셋 준비 완료! (총 {len(dataset)}개 샘플)")


✅ 데이터셋 준비 완료! (총 112개 샘플)


## 예제 8.2. 모델과 토크나이저를 불러와 추론 파이프라인 준비

In [7]:
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# 모델 ID 정의
model_id = "shangrilar/yi-ko-6b-text2sql"

# 4bit 양자화 설정을 포함하여 모델 로드 (VRAM 사용량 최적화)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

# 토크나이저 및 추론 파이프라인 생성
tokenizer = AutoTokenizer.from_pretrained(model_id)
hf_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer)

print("✅ 예제 8.2 성공: Hugging Face 모델 로드 완료")


TypeError: LlamaForCausalLM.__init__() got an unexpected keyword argument 'load_in_4bit'

## 예제 8.3. 배치 크기에 따른 추론 시간 확인

In [ ]:
import time

# 배치 크기 리스트 (교과서 구성과 동일)
for batch_size in [1, 2, 4, 8, 16, 32]:
    start_time = time.time()

    # 전체 데이터셋에 대해 추론 수행
    hf_pipeline(
        dataset['prompt'].tolist(),
        max_new_tokens=128,
        batch_size=batch_size
    )

    # 교과서 원본 출력 형식 유지
    print(f'{batch_size}: {time.time() - start_time}')

print("✅ 예제 8.3 성공: 모든 배치 사이즈 측정 완료")


## 예제 8.4. vLLM 모델 불러오기

### 안내
모델을 여러 번 GPU에 올리기 때문에 CUDA out of memory 에러가 발생할 수 있습니다. 그런 경우 구글 코랩의 런타임 > 세션 다시 시작 후 예제 코드를 실행해주세요.
예제 실행에 데이터셋이 필요한 경우 예제 8.1의 코드를 실행해주세요.

In [ ]:
import gc
import torch
try: del hf_pipeline, model, tokenizer
except: pass
gc.collect()
torch.cuda.empty_cache()
print("✅ VRAM 청소 완료!")


## 예제 8.5. vLLM을 활용한 오프라인 추론 시간 측정

In [ ]:
import time

for max_num_seqs in [1, 2, 4, 8, 16, 32]:
  start_time = time.time()
  llm.llm_engine.scheduler_config.max_num_seqs = max_num_seqs
  sampling_params = SamplingParams(temperature=1, top_p=1, max_tokens=128)
  outputs = llm.generate(dataset['prompt'].tolist(), sampling_params)
  print(f'{max_num_seqs}: {time.time() - start_time}')

## 예제 8.6. 온라인 서빙을 위한 vLLM API 서버 실행

### 안내
모델을 여러 번 GPU에 올리기 때문에 CUDA out of memory 에러가 발생할 수 있습니다. 그런 경우 구글 코랩의 런타임 > 세션 다시 시작 후 예제 코드를 실행해주세요.
예제 실행에 데이터셋이 필요한 경우 예제 8.1의 코드를 실행해주세요.

In [ ]:
!python -m vllm.entrypoints.openai.api_server \
--model shangrilar/yi-ko-6b-text2sql --host 127.0.0.1 --port 8888 --max-model-len 1024

## 예제 8.7. 백그라운드에서 vLLM API 서버 실행하기

In [ ]:
!nohup python -m vllm.entrypoints.openai.api_server \
--model shangrilar/yi-ko-6b-text2sql --host 127.0.0.1 --port 8888 --max-model-len 512 &

## 예제 8.8. API 서버 실행 확인

In [ ]:
!curl http://localhost:8888/v1/models

## 예제 8.9. API 요청

In [ ]:
import json

json_data = json.dumps(
    {"model": "shangrilar/yi-ko-6b-text2sql",
      "prompt": dataset.loc[0, "prompt"],
      "max_tokens": 128,
      "temperature": 1}
    )

!curl http://localhost:8888/v1/completions \
    -H "Content-Type: application/json" \
    -d '{json_data}'

## 예제 8.10. OpenAI 클라이언트를 사용한 API 요청

In [ ]:
from openai import OpenAI

openai_api_key = "EMPTY"
openai_api_base = "http://localhost:8888/v1"
client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)
completion = client.completions.create(model="shangrilar/yi-ko-6b-text2sql",
                                 prompt=dataset.loc[0, 'prompt'], max_tokens=128)
print("생성 결과:", completion.choices[0].text)